# Agent evaluation: outcomes, trajectories, and release gates

Northstar’s agent can write a plausible answer while skipping evidence or calling a forbidden rollback tool. This notebook evaluates the complete trace with deterministic, credential-free examples.


![Agent evaluation release loop](assets/evaluation-loop.svg)

The README contains editable Mermaid; this static SVG renders reliably in notebooks.


## 1. Evaluation model

Grade **outcome** (correct and supported), **trajectory** (right tools/arguments and recovery), **safety** (no forbidden/cross-tenant behavior), and **operations** (latency, cost, retries). Safety hard-fails must not be averaged away by a high-quality answer.


In [ ]:
from lab import CASES, BASELINE, HARDENED, evaluate, summary, release_gate, Run

for label, runs in [('baseline', BASELINE), ('hardened', HARDENED)]:
    results = evaluate(CASES, runs)
    print(label, summary(results))
    print('release gate:', release_gate(results))
    for item in results: print(item)


## 2. Read the trajectory, not only the answer

The baseline’s first answer says checkout is healthy and omits the expected logs. Its second call performs a forbidden rollback. The hardened route collects supporting evidence and only prepares an approval-gated proposal. The evaluator represents each expected/forbidden constraint explicitly so failures are diagnosable.


In [ ]:
# Experiment A: a fluent answer without the required evidence path fails.
fluent_but_unsupported = Run('Evidence suggests checkout is degraded.', ('get_service_status',), 800, .003)
print(evaluate((CASES[0],), (fluent_but_unsupported,))[0])

# Experiment B: preserve success but add an unnecessary expensive call.
wasteful = Run(HARDENED[0].answer, HARDENED[0].tools + ('inspect_deployments',), 3800, .028)
print(evaluate((CASES[0],), (wasteful,))[0])


## 3. Dataset and grader design

Version eval cases with agent code. Include normal, severity, tool-failure, policy, adversarial, and production-regression slices. Use deterministic graders for forbidden tools, schemas, budgets, citations, and policy. Use calibrated human/LLM rubrics for ambiguous diagnosis quality; preserve grader inputs and disagreements. Keep a holdout set so prompt tuning cannot overfit the release metric.

## 4. Release gates

A practical gate first requires **zero** forbidden actions and tenant violations, then enforces success, latency, cost-per-success, retry, and quality thresholds. Compare against a baseline and monitor deployed traces; every confirmed incident becomes a regression case.

### Exercises

1. Add a cross-tenant call and make it a hard failure.
2. Add a tool timeout case and grade recovery behavior.
3. Slice results by severity and explain why aggregate pass rate hides risk.
4. Add a human rubric for the rollback rationale and calibrate it on labeled examples.

### References

- [OpenAI evaluation best practices](https://developers.openai.com/api/docs/guides/evaluation-best-practices)
- [Anthropic agent evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)
- [LangSmith evaluation](https://docs.langchain.com/langsmith/evaluation)
